In [1]:
import mlflow
import dagshub

C:\Users\KIIT0001\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow")


In [3]:

dagshub.init(repo_owner='Aayush10671', repo_name='yt-comment-sentiment-analysis', mlflow=True)
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

Accessing as Aayush10671

Initialized MLflow to track repo "Aayush10671/yt-comment-sentiment-analysis"

Repository Aayush10671/yt-comment-sentiment-analysis initialized!

🏃 View run thundering-rat-828 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0/runs/e0ab1c2ab3e0493080b8ca040d941e3d
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/0


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv("https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv")
df.head(5)

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.dropna(inplace=True)

In [8]:
df = df[~(df['clean_comment'].str.strip() == '')]

In [9]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [10]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [15]:
df.fillna('', inplace=True)

In [20]:
def preprocess_text(text: pd.Series) -> pd.Series:
    # Lowercase
    text = text.str.lower()
    # Remove leading/trailing whitespace
    text = text.str.strip()
    # Replace newlines with spaces
    text = text.str.replace(r'\n', ' ', regex=True)
    # Keep only letters, digits, spaces, and punctuation ! ? .
    text = text.str.replace(r'[^A-Za-z0-9\s!?.]', '', regex=True)
    
    # Stopword removal (excluding negation words)
    stop_words = set(stopwords.words('english')) - {
        'not', 'no', 'but', 'however', 'against', 'nor',
        "don't", "ain't", "aren't", "couldn't", "didn't",
        "doesn't", "hadn't", "hasn't", "haven't", "isn't", "yet"
    }
    text = text.apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    text = text.apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()]))
    
    return text

In [21]:
df['clean_comment'] = preprocess_text(df['clean_comment'])

In [22]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [23]:
vectorizer = CountVectorizer(max_features=1000)

In [25]:
X = vectorizer.fit_transform(df['clean_comment']).toarray()
y = df['category'].values

In [26]:
mlflow.set_experiment("baseline_model_experiment- RF")

2026/07/25 18:34:23 INFO mlflow.tracking.fluent: Experiment with name 'baseline_model_experiment- RF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/92d7501f04ad48779b11351cbc530fca', creation_time=1784984663077, experiment_id='1', last_update_time=1784984663077, lifecycle_stage='active', name='baseline_model_experiment- RF', tags={}, workspace='default'>

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run():
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("max_features", vectorizer.max_features)
    mlflow.log_param("vectorizer_type", "CountVectorizer")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    n_estimators = 100
    max_depth = 10

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    mlflow.sklearn.log_model(model, "model")

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')

    f1 = f1_score(y_test, y_pred, average='weighted')

    mlflow.log_metric("accuracy", accuracy)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    classification_report = classification_report(y_test, y_pred, output_dict=True)
    mlflow.log_dict(classification_report, "classification_report")

    confusion_mat = confusion_matrix(y_test, y_pred)

    mlflow.log_dict({"confusion_matrix": confusion_mat.tolist()}, "confusion_matrix")

    df.to_csv("preprocessed_data.csv", index=False)
    mlflow.log_artifact("preprocessed_data.csv", artifact_path="data")

print("accuracy_score:", accuracy)
print("precision_score:", precision)
print("recall_score:", recall)
print("f1_score:", f1)


2026/07/25 18:41:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/25 18:41:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run orderly-asp-415 at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/1/runs/bf21ffda9a7042d78a738983635831f7
🧪 View experiment at: https://dagshub.com/Aayush10671/yt-comment-sentiment-analysis.mlflow/#/experiments/1
accuracy_score: 0.6290256828373421
precision_score: 0.7119332004501472
recall_score: 0.6290256828373421
f1_score: 0.5556074228418457


In [30]:
df.to_csv("preprocessed_data.csv", index=False)